# Test notebook

The purpose of this notebook is to test an equation and compare them with the baselines: Burton, MBR, and DDM1, 2 and 3. We will also plot each storm and get the metrics for the equation.

The only cell that we have to modify is the following one, where we can change the features, the mode (template or default) and the output directory for the plots.
Raw EQ is the equation that we want to test, the raw version generated from the train_script.py file.

In [1]:
import os

FEATURES = ["P_dyn", "VBs", "epsilon", "DST"]
MODE = "template"  # 'template' or 'default'
OUTPUT_DIR = "template_deriv_features"
RAW_EQS = [
    "g = (#2 * -0.0010713526) * sqrt(#1 + 1.32442); d = square((#1 * 0.01838707) - 0.5912093)",
    "g = #2 * (sqrt(#1 - -1.1162246) * -0.0010842164); d = square((#1 * -0.01812232) + 0.5913376)",
    "g = (sqrt(#1 - -1.1051167) * #2) * -0.0010888062; d = square((#1 * -0.018209398) + 0.59240085)",
]

output_folder = OUTPUT_DIR
# Count number of existing subfolders
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

import sympy as sp
from tqdm import tqdm

from sympy.printing import latex

# Internal module imports
import storm_dates
import baseline_models

# from evaluation_engine import UnifiedModel, simulate_storm, compute_features
from evaluation_engine import EquationModel, simulate_storm
from train_script import load_and_preprocess, compute_features

[juliapkg] Found dependencies: /mnt/data/symbolic-regression-dst-public-repo/.venv/lib/python3.12/site-packages/juliacall/juliapkg.json
[juliapkg] Found dependencies: /mnt/data/symbolic-regression-dst-public-repo/.venv/lib/python3.12/site-packages/juliapkg/juliapkg.json
[juliapkg] Found dependencies: /mnt/data/symbolic-regression-dst-public-repo/.venv/lib/python3.12/site-packages/pysr/juliapkg.json
[juliapkg] Locating Julia 1.10.3 - 1.11
[juliapkg] Querying Julia versions from https://julialang-s3.julialang.org/bin/versions.json
[juliapkg] Using Julia 1.11.9 at /mnt/data/symbolic-regression-dst-public-repo/.venv/julia_env/pyjuliapkg/install/bin/julia
[juliapkg] Using Julia project at /mnt/data/symbolic-regression-dst-public-repo/.venv/julia_env
[juliapkg] Writing Project.toml:
           | [deps]
           | PythonCall = "6099a3de-0909-46bc-b1f4-468b9a2dfc0d"
           | OpenSSL_jll = "458c3c95-2e84-50aa-8efc-19380b2a3a95"
           | SymbolicRegression = "8254be44-1295-4e6a-a16d-46

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
    Updating `/mnt/data/symbolic-regression-ldi-internal/.venv/julia_env/Project.toml`
⌅ [6099a3de] + PythonCall v0.9.26
⌅ [8254be44] + SymbolicRegression v1.11.3
⌅ [458c3c95] + OpenSSL_jll v3.0.20+0
  [9e88b42a] ~ Serialization ⇒ v1.11.0
    Updating `/mnt/data/symbolic-regression-ldi-internal/.venv/julia_env/Manifest.toml`
  [47edcb42] + ADTypes v1.24.0
  [79e6a3ab] + Adapt v4.7.0
  [66dad0bd] + AliasTables v1.1.3
  [4fba245c] + ArrayInterface v7.30.1
  [d360d2e6] + ChainRulesCore v1.26.1
  [bbf7d656] + CommonSubexpressions v0.3.1
  [34da2185] + Compat v4.18.1
  [992eb4ea] + CondaPkg v0.2.36
  [187b0558] + ConstructionBase v1.6.0
  [9a962f9c] + DataAPI v1.16.0
  [864edb3b] + DataStructures v0.19.6
  [e2d170a0] + DataValueInterfaces v1.0.0
  [163ba53b] + DiffResults v1.1.0
  [b552c78f] + DiffRules v1.16.0
  [a0c0ee7d] + DifferentiationInterface v0.7.21
  [8d63f2c5] + DispatchDoctor v0.4.28
  [

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


/mnt/data/symbolic-regression-dst-public-repo/.venv/lib/python3.12/site-packages/spacepy/time.py:2448: UserWarning: Leapseconds may be out of date. Use spacepy.toolbox.update(leapsecs=True)
  _read_leaps()


In [3]:
raw_data = load_and_preprocess()
data = compute_features(raw_data)

In [4]:
def predict_and_plot_storm(model, eqs, start, end, storm_df, storm_id, save_path):
    # 1. Generate Predictions
    y_true = storm_df[start:end]["DST"].values
    colors = ["blue", "yellow", "green", "orange", "purple", "cyan"]
    res_eqs = []
    string_title = f"Storm {storm_id} Reconstruction\n"
    metrics_info = []
    
    
    for eq_index, eq in enumerate(eqs):
        res_eq = simulate_storm(model[eq], storm_df)
        res_eq = res_eq[start:end]["DST_pred"].values
        m_eq = baseline_models.get_all_metrics_dict(y_true, res_eq)
        metrics_info.append(m_eq)
        string_title += f"Evaluation for Equation {eq_index + 1} ({colors[eq_index]}): ${model[eq].latex_str()}$ \n"
    
        res_eqs.append(res_eq)

    # 2. Calculate Metrics
    
    # 3. Setup Figure (3 Columns)
    fig, axs = plt.subplots(1, 3, figsize=(24, 7), constrained_layout=True)
    fig.suptitle(
        string_title, fontsize=18
    )
    # Column 1: Time Series
    axs[0].plot(
        storm_df[start:end].index,
        y_true,
        color="black",
        label="Observed",
        alpha=0.6,
        linewidth=2,
    )
    
    for eq_index, res_eq in enumerate(res_eqs):
    
        axs[0].plot(
            storm_df[start:end].index,
            res_eqs[eq_index],
            color=colors[eq_index],
            linestyle="--",
            label=f"Equation {eq_index + 1}",
            linewidth=1.5,
        )
    
    axs[0].set_title(f"Storm {storm_id} Reconstruction", fontsize=18)
    axs[0].tick_params(axis='both', which='major', labelsize=14)
    axs[0].tick_params(axis='both', which='minor', labelsize=10)
    axs[0].legend(fontsize = 16)
    axs[0].grid(True)
    axs[0].set_xlim(start, end)
    axs[0].set_xlabel("Date", fontsize=16)
    axs[0].set_ylabel("Dst (nT)", fontsize=16)
    axs[0].xaxis.set_major_locator(MultipleLocator(2))
    
    diffs = []
    
    for eq_index, res_eq in enumerate(res_eqs):
        diff_eq = res_eq - y_true
        diffs.append(diff_eq)

        axs[1].plot(storm_df[start:end].index, diff_eq, color=colors[eq_index], label=f"Equation {eq_index + 1} Error")
    
    axs[1].axhline(0, color="black", linestyle="--")

    title_metrics = (
        f"Error Comparison\n"       
    )
    
    for eq_index, m_eq in enumerate(metrics_info):
        title_metrics += f"Eq {eq_index + 1}: MAE={m_eq['MAE']:.2f}, RMSE={m_eq['RMSE']:.2f}, R²={m_eq['R2']:.3f}, BFE={m_eq['BFE']:.3f}\n"
        
        
    axs[1].set_title(title_metrics, fontsize=18)
    axs[1].set_ylabel("Error (nT)", fontsize=16)
    axs[1].set_xlabel("Date", fontsize=16)
    axs[1].legend(fontsize=16)
    axs[1].grid(True)
    axs[1].set_xlim(start, end)
    axs[1].tick_params(axis='both', which='major', labelsize=14)
    axs[1].tick_params(axis='both', which='minor', labelsize=10)
    
    axs[1].xaxis.set_major_locator(MultipleLocator(2))

    # Column 3: BFE
    baseline_models.plot_evaluation_bfe_multi(
        axs[2],
        y_true,
        res_eqs,
        [f"Equation {i+1}" for i in range(len(eqs))],
        [colors[i] for i in range(len(eqs))],
        fontsize = 16   
    )

    plt.savefig(save_path)
    plt.close()


In [5]:
def save_prediction_data(model, eqs, start, end, storm_df, output_path):
    """
    Generates and saves a CSV with observed and predicted DST and dDST/dt.
    """
    # 1. Observed Data
    # Real dDST is calculated as the difference to the next hour
    real_dst = storm_df[start:end]["DST"].values
    real_ddst = storm_df[start:end]["DST"].diff().shift(-1).values

    # 2. Equation Predictions
    # We need the iterative predictions for DST
    pred_dst_eqs = []
    for eq_index, eq in enumerate(eqs):
        pred_dst_eq = simulate_storm(model[eq], storm_df)
        if model[eq].is_template:
            pred_dst_eq = pred_dst_eq[start:end][
                ["DST_pred", "dDST", "injection_component", "decay_component"]
            ]
        else:
            pred_dst_eq = pred_dst_eq[start:end][["DST_pred", "dDST"]]
        pred_dst_eqs.append(pred_dst_eq)
        
    # 3. Baseline Predictions (Burton & OBM)
    
    # 4. Construct Comprehensive DataFrame

    if model[eq].is_template:
        results_df = pd.DataFrame(
            {
                "Timestamp": storm_df[start:end].index,
                "Observed_DST": real_dst,
                "Real_dDST_dt": real_ddst,
                "Pred_DST_Equation": pred_dst_eq["DST_pred"].values,
                "Pred_dDST_dt_Equation": pred_dst_eq["dDST"].values,
                "Injection_Component": pred_dst_eq["injection_component"].values,
                "Decay_Component": pred_dst_eq["decay_component"].values,
                
            }
        ).set_index("Timestamp")
        
        for eq_index, pred_dst_eq in enumerate(pred_dst_eqs):
            results_df[f"Pred_DST_Equation_{eq_index+1}"] = pred_dst_eq["DST_pred"].values
            results_df[f"Pred_dDST_dt_Equation_{eq_index+1}"] = pred_dst_eq["dDST"].values
            results_df[f"Injection_Component_{eq_index+1}"] = pred_dst_eq["injection_component"].values
            results_df[f"Decay_Component_{eq_index+1}"] = pred_dst_eq["decay_component"].values
        
    else:
        results_df = pd.DataFrame(
            {
                "Timestamp": storm_df[start:end].index,
                "Observed_DST": real_dst,
                "Real_dDST_dt": real_ddst,                
            }
        ).set_index("Timestamp")
        
        for eq_index, pred_dst_eq in enumerate(pred_dst_eqs):
            results_df[f"Pred_DST_Equation_{eq_index+1}"] = pred_dst_eq["DST_pred"].values
            results_df[f"Pred_dDST_dt_Equation_{eq_index+1}"] = pred_dst_eq["dDST"].values


    results_df.to_csv(output_path)
    return results_df

## Test storms

In [6]:
storms = []
storm_indices = []
models = {}

for RAW_EQ in RAW_EQS:
    model = EquationModel(RAW_EQ, FEATURES, is_template=MODE == "template")
    models[RAW_EQ] = model
    
    
test_storms = storm_dates.TEST_STORMS_SYMBOLIC_REGRESSION

for sd, ed, storm_id in tqdm(test_storms):
    start = pd.to_datetime(sd)
    end = pd.to_datetime(ed)
    storm_df = data[
        start - pd.DateOffset(hours=1) : end + pd.DateOffset(hours=1)
    ].copy()
    if storm_df.empty:
        continue

    file_name = f"storm_{storm_id}.png"
    predict_and_plot_storm(
        models,
        RAW_EQS,
        start,
        end,
        storm_df,
        storm_id,
        os.path.join(OUTPUT_DIR, file_name),
    )

    csv_name = f"data_storm_{storm_id}.csv"
    storms.append(
        save_prediction_data(
            models, RAW_EQS, start, end, storm_df, os.path.join(OUTPUT_DIR, csv_name)
        )
    )
    storm_indices.append(storm_id)
    
with open(os.path.join(OUTPUT_DIR, 'equation.txt'), 'a') as f:
    f.write(f'Equation: {RAW_EQ}\n')            
    f.write(f'LaTeX: {latex(models[RAW_EQ].latex_str())}\n')

  0%|          | 0/20 [00:00<?, ?it/s]

100%|██████████| 20/20 [00:20<00:00,  1.03s/it]


In [7]:
storms[0].columns

Index(['Observed_DST', 'Real_dDST_dt', 'Pred_DST_Equation',
       'Pred_dDST_dt_Equation', 'Injection_Component', 'Decay_Component',
       'Pred_DST_Equation_1', 'Pred_dDST_dt_Equation_1',
       'Injection_Component_1', 'Decay_Component_1', 'Pred_DST_Equation_2',
       'Pred_dDST_dt_Equation_2', 'Injection_Component_2', 'Decay_Component_2',
       'Pred_DST_Equation_3', 'Pred_dDST_dt_Equation_3',
       'Injection_Component_3', 'Decay_Component_3'],
      dtype='object')

In [8]:
metrics = ["RMSE", "MAE", "R2", "CC", "BFE"]
equations = [f"Equation {i+1}" for i in range(len(RAW_EQS))]

columns = [f"{eq}_{metric}" for eq in equations for metric in metrics]

summary_df = pd.DataFrame(
    columns=["Storm Index"] + columns,
)

for storm_index, storm in enumerate(storms):
    summary_df.loc[storm_indices[storm_index], "Storm Index"] = storm_indices[storm_index]


for storm_index, storm in enumerate(storms):
    y_true = storm["Observed_DST"].values
    
    for eq_index in range(len(RAW_EQS)):
    
        res_eq = storm[f"Pred_DST_Equation_{eq_index+1}"].values
    

        m_eq = baseline_models.get_all_metrics_dict(y_true, res_eq)
        
        for metric in metrics:
            summary_df.loc[storm_indices[storm_index], f"Equation {eq_index+1}_{metric}"] = m_eq[metric]
        
        

summary_df.loc[len(summary_df)] = ["Mean", *summary_df[columns].mean().values]


global_data = pd.concat(storms, ignore_index=True)
y_true = global_data["Observed_DST"].values

summary_df.loc[len(summary_df), "Storm Index"] = 'Global'

for eq_index in range(len(RAW_EQS)):
    res_eq = global_data[f"Pred_DST_Equation_{eq_index+1}"].values
    m_eq = baseline_models.get_all_metrics_dict(y_true, res_eq)    
    for metric in metrics:
        summary_df.loc[len(summary_df) - 1, f"Equation {eq_index+1}_{metric}"] = m_eq[metric]


display(summary_df)

,Storm Index,Equation 1_RMSE,Equation 1_MAE,Equation 1_R2,Equation 1_CC,Equation 1_BFE,Equation 2_RMSE,Equation 2_MAE,Equation 2_R2,Equation 2_CC,Equation 2_BFE,Equation 3_RMSE,Equation 3_MAE,Equation 3_R2,Equation 3_CC,Equation 3_BFE
54,54,9.08245,7.150428,0.796405,0.922839,11.620567,9.133584,7.232591,0.794106,0.921149,11.626461,9.19187,7.301247,0.79147,0.920786,11.646322
55,55,17.429458,13.783357,0.748936,0.886955,19.558619,17.564804,13.841243,0.745022,0.885049,19.784708,17.584147,13.887456,0.74446,0.885318,19.750457
56,56,13.276276,10.962537,0.613082,0.889956,15.586261,13.524116,11.247863,0.598501,0.888208,15.825734,13.528827,11.279733,0.598222,0.888566,15.788001
57,57,9.5797,7.665862,0.76669,0.899931,12.124595,9.57895,7.65709,0.766726,0.899847,12.266801,9.542605,7.634365,0.768493,0.900465,12.214531
58,58,9.165543,6.390252,0.797016,0.927846,15.584912,9.21504,6.349188,0.794818,0.930432,15.791776,9.191613,6.313452,0.79586,0.931173,15.77743
59,59,12.207318,10.24129,0.849938,0.955928,9.432197,12.082922,10.088617,0.852981,0.955455,9.451668,11.975463,9.989224,0.855584,0.955646,9.409921
60,60,17.617883,11.828793,0.777722,0.934481,26.807645,18.391144,11.914897,0.757782,0.9305,28.591611,18.421335,11.915238,0.756986,0.930995,28.684397
61,61,10.679534,8.252146,0.932006,0.968053,14.5531,10.784213,8.343296,0.930666,0.96835,14.538597,10.823833,8.392287,0.930156,0.968559,14.457715
62,62,11.394979,9.122644,0.698127,0.886644,12.820999,11.196326,8.935398,0.708561,0.889372,13.001519,11.177927,8.930614,0.709518,0.888333,13.034045
63,63,14.975924,12.167919,0.837054,0.944592,18.193426,14.829444,12.014736,0.840226,0.945314,17.878158,14.927598,12.134237,0.838104,0.945052,17.99657


In [9]:
print(summary_df.to_latex(index=False, float_format="%.3f").replace("_", " ").replace("Equation ", "Eq ").replace("Storm Index", "Storm"))

\begin{tabular}{llllllllllllllll}
\toprule
Storm & Eq 1 RMSE & Eq 1 MAE & Eq 1 R2 & Eq 1 CC & Eq 1 BFE & Eq 2 RMSE & Eq 2 MAE & Eq 2 R2 & Eq 2 CC & Eq 2 BFE & Eq 3 RMSE & Eq 3 MAE & Eq 3 R2 & Eq 3 CC & Eq 3 BFE \\
\midrule
54 & 9.082 & 7.150 & 0.796 & 0.923 & 11.621 & 9.134 & 7.233 & 0.794 & 0.921 & 11.626 & 9.192 & 7.301 & 0.791 & 0.921 & 11.646 \\
55 & 17.429 & 13.783 & 0.749 & 0.887 & 19.559 & 17.565 & 13.841 & 0.745 & 0.885 & 19.785 & 17.584 & 13.887 & 0.744 & 0.885 & 19.750 \\
56 & 13.276 & 10.963 & 0.613 & 0.890 & 15.586 & 13.524 & 11.248 & 0.599 & 0.888 & 15.826 & 13.529 & 11.280 & 0.598 & 0.889 & 15.788 \\
57 & 9.580 & 7.666 & 0.767 & 0.900 & 12.125 & 9.579 & 7.657 & 0.767 & 0.900 & 12.267 & 9.543 & 7.634 & 0.768 & 0.900 & 12.215 \\
58 & 9.166 & 6.390 & 0.797 & 0.928 & 15.585 & 9.215 & 6.349 & 0.795 & 0.930 & 15.792 & 9.192 & 6.313 & 0.796 & 0.931 & 15.777 \\
59 & 12.207 & 10.241 & 0.850 & 0.956 & 9.432 & 12.083 & 10.089 & 0.853 & 0.955 & 9.452 & 11.975 & 9.989 & 0.856 & 0.956 

In [10]:
display(summary_df[['Storm Index', 'Equation 1_BFE', 'Equation 2_BFE', 'Equation 3_BFE']].set_index('Storm Index'))

,Equation 1_BFE,Equation 2_BFE,Equation 3_BFE
Storm Index,,,
54,11.620567,11.626461,11.646322
55,19.558619,19.784708,19.750457
56,15.586261,15.825734,15.788001
57,12.124595,12.266801,12.214531
58,15.584912,15.791776,15.77743
59,9.432197,9.451668,9.409921
60,26.807645,28.591611,28.684397
61,14.5531,14.538597,14.457715
62,12.820999,13.001519,13.034045


## Train storms

In [11]:
storms = []
storm_indices = []
models = {}

for RAW_EQ in RAW_EQS:
    model = EquationModel(RAW_EQ, FEATURES, is_template=MODE == "template")
    models[RAW_EQ] = model
    
    
test_storms = storm_dates.TRAIN_STORMS_SYMBOLIC_REGRESSION

for sd, ed, storm_id in tqdm(test_storms):
    start = pd.to_datetime(sd)
    end = pd.to_datetime(ed)
    storm_df = data[
        start - pd.DateOffset(hours=1) : end + pd.DateOffset(hours=1)
    ].copy()
    if storm_df.empty:
        continue

    file_name = f"storm_{storm_id}.png"
    predict_and_plot_storm(
        models,
        RAW_EQS,
        start,
        end,
        storm_df,
        storm_id,
        os.path.join(OUTPUT_DIR, file_name),
    )

    csv_name = f"data_storm_{storm_id}.csv"
    storms.append(
        save_prediction_data(
            models, RAW_EQS, start, end, storm_df, os.path.join(OUTPUT_DIR, csv_name)
        )
    )
    storm_indices.append(storm_id)


100%|██████████| 53/53 [00:55<00:00,  1.04s/it]


In [12]:
metrics = ["RMSE", "MAE", "R2", "CC", "BFE"]
equations = [f"Equation {i+1}" for i in range(len(RAW_EQS))]

columns = [f"{eq}_{metric}" for eq in equations for metric in metrics]

summary_df = pd.DataFrame(
    columns=["Storm Index"] + columns,
)

for storm_index, storm in enumerate(storms):
    summary_df.loc[len(summary_df), "Storm Index"] = storm_indices[storm_index]


for storm_index, storm in enumerate(storms):
    y_true = storm["Observed_DST"].values
    
    for eq_index in range(len(RAW_EQS)):
    
        res_eq = storm[f"Pred_DST_Equation_{eq_index+1}"].values
    

        m_eq = baseline_models.get_all_metrics_dict(y_true, res_eq)
        
        for metric in metrics:
            summary_df.loc[storm_indices[storm_index], f"Equation {eq_index+1}_{metric}"] = m_eq[metric]
        
        

display(summary_df.mean())

summary_df.loc[len(summary_df)] = ["Mean", *summary_df[columns].mean().values]

global_data = pd.concat(storms, ignore_index=True)
y_true = global_data["Observed_DST"].values

summary_df.loc[len(summary_df), "Storm Index"] = 'Global'



for eq_index in range(len(RAW_EQS)):
    res_eq = global_data[f"Pred_DST_Equation_{eq_index+1}"].values
    m_eq = baseline_models.get_all_metrics_dict(y_true, res_eq)    
    for metric in metrics:
        summary_df.loc[len(summary_df) - 1, f"Equation {eq_index+1}_{metric}"] = m_eq[metric]


display(summary_df)

Storm Index             27.0
Equation 1_RMSE    15.372218
Equation 1_MAE     11.877795
Equation 1_R2       0.670062
Equation 1_CC       0.903021
Equation 1_BFE     16.737971
Equation 2_RMSE    15.368584
Equation 2_MAE     11.854889
Equation 2_R2       0.667983
Equation 2_CC       0.903389
Equation 2_BFE     16.720738
Equation 3_RMSE    15.354028
Equation 3_MAE     11.854797
Equation 3_R2       0.669695
Equation 3_CC        0.90333
Equation 3_BFE     16.701925
dtype: object

,Storm Index,Equation 1_RMSE,Equation 1_MAE,Equation 1_R2,Equation 1_CC,Equation 1_BFE,Equation 2_RMSE,Equation 2_MAE,Equation 2_R2,Equation 2_CC,Equation 2_BFE,Equation 3_RMSE,Equation 3_MAE,Equation 3_R2,Equation 3_CC,Equation 3_BFE
0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,20.213585,17.117328,0.640361,0.871804,16.822423,20.41542,17.287926,0.633143,0.872032,17.159075,20.50151,17.406734,0.630042,0.871133,17.186395
2,3,22.835546,17.477713,-0.337692,0.578832,19.405386,23.023223,17.644304,-0.35977,0.576921,19.584744,22.949039,17.583713,-0.351021,0.577078,19.519541
3,4,11.215986,8.793026,0.879061,0.949642,8.793067,10.957405,8.588848,0.884574,0.952284,8.530354,10.970735,8.601882,0.884293,0.951741,8.577101
4,5,17.213986,14.144183,0.77824,0.920668,19.953851,17.264461,14.17047,0.776938,0.921216,20.062491,17.250677,14.201701,0.777294,0.920853,19.919848
5,6,18.665306,14.714571,0.766902,0.907633,24.272886,18.811398,14.733568,0.763239,0.907146,24.423398,18.759678,14.761246,0.764539,0.907565,24.314309
6,7,15.532397,12.371153,0.645368,0.883474,16.965733,15.56861,12.437606,0.643712,0.88261,16.892376,15.509408,12.420285,0.646417,0.881884,16.838156
7,8,10.465127,8.609895,0.808674,0.906307,7.050362,10.349696,8.514791,0.812871,0.909175,6.958053,10.366637,8.549774,0.812258,0.908852,6.963698
8,9,12.263987,9.014927,0.847011,0.962559,13.26364,12.348562,9.06949,0.844894,0.962723,13.511569,12.318126,9.071194,0.845658,0.962683,13.3791
9,10,12.576511,9.406606,0.692773,0.918247,11.592131,12.848755,9.606697,0.679328,0.916641,11.848807,12.746874,9.512244,0.684394,0.916899,11.756749


In [13]:
display(summary_df[['Storm Index', 'Equation 1_BFE', 'Equation 2_BFE', 'Equation 3_BFE']].set_index('Storm Index'))

,Equation 1_BFE,Equation 2_BFE,Equation 3_BFE
Storm Index,,,
1,NaN,NaN,NaN
2,16.822423,17.159075,17.186395
3,19.405386,19.584744,19.519541
4,8.793067,8.530354,8.577101
5,19.953851,20.062491,19.919848
6,24.272886,24.423398,24.314309
7,16.965733,16.892376,16.838156
8,7.050362,6.958053,6.963698
9,13.26364,13.511569,13.3791


In [14]:

storms = range(54, 74)
parent_folder = 'template-derived-figures'

for storm_number in storms:
    # We use f-strings with double {{ }} to escape the LaTeX braces
    # and single { } for the Python variables.
    latex_code = f"""
\\begin{{figure}}[ht]
    \\centering
    \\includegraphics[width=\\textwidth]{{{parent_folder}/storm_{storm_number}.png}}
    \\caption{{Reconstruction of storm {storm_number} using the Equations generated from the templated symbolic regression with the derived features}}\\label{{fig:template-storm-{storm_number}}}
\\end{{figure}}
"""
    print(latex_code)   


\begin{figure}[ht]
    \centering
    \includegraphics[width=\textwidth]{template-derived-figures/storm_54.png}
    \caption{Reconstruction of storm 54 using the Equations generated from the templated symbolic regression with the derived features}\label{fig:template-storm-54}
\end{figure}


\begin{figure}[ht]
    \centering
    \includegraphics[width=\textwidth]{template-derived-figures/storm_55.png}
    \caption{Reconstruction of storm 55 using the Equations generated from the templated symbolic regression with the derived features}\label{fig:template-storm-55}
\end{figure}


\begin{figure}[ht]
    \centering
    \includegraphics[width=\textwidth]{template-derived-figures/storm_56.png}
    \caption{Reconstruction of storm 56 using the Equations generated from the templated symbolic regression with the derived features}\label{fig:template-storm-56}
\end{figure}


\begin{figure}[ht]
    \centering
    \includegraphics[width=\textwidth]{template-derived-figures/storm_57.png}
    \captio